# catboost_postprocess_v1_fix.ipynb
仅基于**CatBoost 初版**进行**后处理**（不重训、不加 LGBM）。

**修复点**：当 `grade` 或 `has_stm` 不存在时自动构造：
- `grade`：从 `level` 的首字母推断（如 `B3` → `B`）
- `has_stm`：若无该列，则根据是否在 `*_statement_feature*.csv` 中出现来判定
- `time_bin`：若无 `issue_time_days`，使用 `issue_time`（Unix）等频分箱

**输出**：三套提交文件（segment isotonic / rank 融合 / stack-LR）。

In [25]:

# ================= 配置 =================
OUT_DIR = "outputs_postprocess_v1"
OOF_PATH_CANDIDATES = [
    "outputs/oof_predictions.csv",
    "oof_predictions.csv",
    "outputs_timeaware_v2/oof_timeaware.csv",
    "outputs_v1/oof_predictions.csv"
]
TEST_AA_PRED_PATHS = [
    "outputs/test_aa_pred_cat.csv",
]
TEST_AB_PRED_PATHS = [
    "outputs/test_pred_catboost.csv"
]

# 可能存在的流水特征文件（用于构造 has_stm）
STM_TRAIN_PATHS = ["train_statement_feature_v2p.csv", "train_statement_feature.csv"]
STM_TESTAA_PATHS = ["testaa_statement_feature_v2p.csv", "testaa_statement_feature.csv"]
STM_TESTAB_PATHS = ["testab_statement_feature_v2p.csv", "testab_statement_feature.csv"]

MIN_SEG_SAMPLES = 300
TIME_BIN_QUANTILES = 5
RANDOM_STATE = 1337


In [26]:

import os, numpy as np, pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

os.makedirs(OUT_DIR, exist_ok=True)

def _first_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    return None

# -------- 基础解析：训练DF / OOF / 测试DF & 预测 --------
def resolve_train_df():
    for name in ['tr','train','df_train','train_df']:
        if name in globals():
            df = globals()[name]
            if isinstance(df, pd.DataFrame) and {'id','label'}.issubset(df.columns):
                return df
    if os.path.exists('train.csv'):
        return pd.read_csv('train.csv')
    raise RuntimeError("未找到训练集 DataFrame，也无法读取 train.csv")

def resolve_features(df):
    if 'features' in globals():
        cand = list(globals()['features'])
        return [c for c in cand if c in df.columns]
    return [c for c in df.columns if c not in ('id','label')]

def resolve_oof(tr_df):
    for name in ['oof','oof_pred','oof_proba','oof_predictions']:
        if name in globals():
            arr = np.asarray(globals()[name], dtype=float).reshape(-1)
            if len(arr) == len(tr_df):
                return arr
    p = _first_existing(OOF_PATH_CANDIDATES)
    if p:
        df = pd.read_csv(p)
        for col in ['oof_pred','oof','prob','pred','oof_cat','oof_timeaware','oof_blend']:
            if col in df.columns:
                if 'id' in df.columns:
                    m = tr_df[['id']].merge(df[['id',col]], on='id', how='left')
                    if m[col].notna().sum() > 0:
                        return m[col].fillna(m[col].mean()).values
                return df[col].values
    raise RuntimeError("未找到 OOF 预测（既没有内存变量，也没有常见 CSV）")

def resolve_test_df(tag):
    # 优先使用内存中的 DataFrame
    name_map = {'aa':['te_aa','test_aa','df_test_aa'], 'ab':['te_ab','test_ab','df_test_ab','test']}
    for nm in name_map[tag]:
        if nm in globals() and isinstance(globals()[nm], pd.DataFrame):
            return globals()[nm]
    # 回退到原始 test CSV（只为拿 id/元特征）
    raw = 'testaa.csv' if tag=='aa' else 'testab.csv'
    if os.path.exists(raw):
        return pd.read_csv(raw)
    return None

def resolve_test_pred(tag):
    # 内存变量
    name_map = {'aa':['pred_test_aa','testaa_pred','pred_aa'], 'ab':['pred_test_ab','testab_pred','pred_ab']}
    for nm in name_map[tag]:
        if nm in globals():
            return np.asarray(globals()[nm], dtype=float).reshape(-1)
    # CSV
    paths = TEST_AA_PRED_PATHS if tag=='aa' else TEST_AB_PRED_PATHS
    p = _first_existing(paths)
    if p:
        df = pd.read_csv(p)
        for col in ['prob','pred','score','label']:
            if col in df.columns:
                return df[col].values
    return None

# -------- 元特征：grade/term/has_stm/level/time_bin --------
def read_stm_ids(paths):
    p = _first_existing(paths)
    if not p: 
        return set()
    try:
        df = pd.read_csv(p, usecols=['id'])
    except Exception:
        df = pd.read_csv(p)
        if 'id' not in df.columns:
            return set()
        df = df[['id']]
    return set(df['id'].astype(str).unique())

STM_IDS_TRAIN = read_stm_ids(STM_TRAIN_PATHS)
STM_IDS_TESTAA = read_stm_ids(STM_TESTAA_PATHS)
STM_IDS_TESTAB = read_stm_ids(STM_TESTAB_PATHS)

def ensure_meta(df, split_tag='train'):
    meta = pd.DataFrame(index=df.index)
    # level/grade
    if 'grade' in df.columns:
        meta['grade'] = df['grade'].astype(str)
    elif 'level' in df.columns:
        lv = df['level'].fillna('NA').astype(str)
        meta['grade'] = lv.str[0].fillna('NA')
        meta['level'] = lv
    else:
        meta['grade'] = 'NA'
    # term
    if 'term' in df.columns:
        meta['term'] = df['term'].astype(str)
    else:
        meta['term'] = 'NA'
    # has_stm
    if 'has_stm' in df.columns:
        meta['has_stm'] = df['has_stm'].astype(int).astype(str)
    else:
        if split_tag=='train':
            meta['has_stm'] = df['id'].astype(str).isin(STM_IDS_TRAIN).map({True:'1', False:'0'})
        elif split_tag=='aa':
            meta['has_stm'] = df['id'].astype(str).isin(STM_IDS_TESTAA).map({True:'1', False:'0'})
        else:
            meta['has_stm'] = df['id'].astype(str).isin(STM_IDS_TESTAB).map({True:'1', False:'0'})
    # time_bin（优先 issue_time_days，否则用 issue_time Unix）
    if 'issue_time_days' in df.columns:
        s = df['issue_time_days']
        try:
            meta['time_bin'] = pd.qcut(s, q=TIME_BIN_QUANTILES, duplicates='drop').astype(str)
        except Exception:
            meta['time_bin'] = 'ALL'
    elif 'issue_time' in df.columns:
        try:
            dt = pd.to_datetime(df['issue_time'], unit='s', utc=True, errors='coerce').astype('int64')
            meta['time_bin'] = pd.qcut(dt, q=TIME_BIN_QUANTILES, duplicates='drop').astype(str)
        except Exception:
            meta['time_bin'] = 'ALL'
    else:
        meta['time_bin'] = 'ALL'
    return meta

def pick_seg_cols(meta_df):
    # 按优先级选择存在的列（至少 1 个；否则只用全局）
    pref = ['grade','has_stm','term','time_bin','level']
    cols = [c for c in pref if c in meta_df.columns]
    if len(cols) >= 2:
        return cols[:2]  # 使用前两个，稳妥
    elif len(cols) == 1:
        return cols
    else:
        return []  # 无可用分段时，回退全局


In [27]:

# -------- 工具函数 --------
def to_logit(p, eps=1e-6):
    p = np.clip(p, eps, 1-eps)
    return np.log(p/(1-p))

def to_rank01(x):
    r = pd.Series(x).rank(method='average').values
    return (r - r.min()) / (r.max() - r.min() + 1e-12)

def save_sub(df, prob, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    pd.DataFrame({'id': df['id'], 'prob': prob}).to_csv(path, index=False, encoding='utf-8')
    print("[SAVE]", path)


In [28]:

# -------- 方案A：分段 Isotonic --------
def fit_segment_isotonic(y, p, meta_df, min_samples=300, seg_cols=('grade','has_stm')):
    seg_cols = [c for c in seg_cols if c in meta_df.columns]
    models = {}
    global_iso = IsotonicRegression(out_of_bounds='clip')
    global_iso.fit(p, y)
    if len(seg_cols) == 0:
        return models, global_iso, seg_cols
    seg_key = meta_df[seg_cols].astype(str).agg('|'.join, axis=1)
    for k, idx in pd.Series(range(len(y))).groupby(seg_key):
        idx = idx.values
        if len(idx) < min_samples:
            continue
        iso = IsotonicRegression(out_of_bounds='clip')
        iso.fit(p[idx], y[idx])
        models[k] = iso
    return models, global_iso, seg_cols

def apply_segment_isotonic(p, meta_df, models_map, global_iso, seg_cols):
    if len(seg_cols) == 0:
        return global_iso.predict(p)
    seg_key = meta_df[seg_cols].astype(str).agg('|'.join, axis=1)
    out = np.zeros_like(p, dtype=float)
    for k, idx in pd.Series(range(len(p))).groupby(seg_key):
        idx = idx.values
        iso = models_map.get(k, global_iso)
        out[idx] = iso.predict(p[idx])
    return out


In [29]:

# -------- 方案B：Stack LR（logit(p)+meta one-hot） --------
from sklearn.metrics import roc_auc_score
def fit_stack_lr(y, p, meta_df):
    X = pd.DataFrame({'logit_p': to_logit(p)})
    X = pd.concat([X, pd.get_dummies(meta_df, drop_first=True)], axis=1)
    pipe = Pipeline([
        ("scaler", StandardScaler(with_mean=False)),
        ("lr", LogisticRegression(C=1.0, solver="lbfgs", max_iter=2000))
    ])
    pipe.fit(X, y)
    return pipe, X.columns.tolist()

def apply_stack_lr(p, meta_df, pipe, train_cols):
    X = pd.DataFrame({'logit_p': to_logit(p)})
    X = pd.concat([X, pd.get_dummies(meta_df, drop_first=True)], axis=1)
    for c in train_cols:
        if c not in X.columns: X[c] = 0
    X = X[train_cols]
    return pipe.predict_proba(X)[:,1]


In [30]:

# ================= 主流程 =================
# 训练集与 OOF
tr = resolve_train_df()
features = resolve_features(tr)
oof_base = resolve_oof(tr)
y_true = tr['label'].values.astype(int)

# 元特征（稳妥构造）
meta_tr = ensure_meta(tr, 'train')
seg_cols = pick_seg_cols(meta_tr)

print("[INFO] 使用分段列：", seg_cols if len(seg_cols)>0 else "无（仅全局 isotonic）")
print("[INFO] OOF base AUC =", roc_auc_score(y_true, oof_base))

# A) 分段 Isotonic
iso_models, iso_global, seg_cols_used = fit_segment_isotonic(y_true, oof_base, meta_tr, min_samples=MIN_SEG_SAMPLES, seg_cols=tuple(seg_cols) if len(seg_cols)>0 else ())
oof_iso = apply_segment_isotonic(oof_base, meta_tr, iso_models, iso_global, seg_cols_used)
print("[A] Segment Isotonic OOF AUC =", roc_auc_score(y_true, oof_iso))

# A-rank) rank 融合（raw vs iso）
best_w, best_auc = 0.5, -1.0
oof_raw_r = to_rank01(oof_base)
oof_iso_r = to_rank01(oof_iso)
for w in np.linspace(0,1,21):
    auc = roc_auc_score(y_true, w*oof_raw_r + (1-w)*oof_iso_r)
    if auc > best_auc: best_auc, best_w = auc, float(w)
print(f"[A-rank] best_w={best_w:.2f} | OOF AUC={best_auc:.6f}")

# B) Stack LR
stack_model, stack_cols = fit_stack_lr(y_true, oof_base, meta_tr)
# 构造训练矩阵（与 apply 对齐）
X_tr_stack = pd.concat([pd.DataFrame({'logit_p': to_logit(oof_base)}), pd.get_dummies(meta_tr, drop_first=True)], axis=1)
for c in stack_cols:
    if c not in X_tr_stack.columns: X_tr_stack[c] = 0
X_tr_stack = X_tr_stack[stack_cols]
oof_stack = stack_model.predict_proba(X_tr_stack)[:,1]
print("[B] Stack LR OOF AUC =", roc_auc_score(y_true, oof_stack))

# ========= 生成 test 提交 =========
for tag in ['aa','ab']:
    df_te = resolve_test_df(tag)
    if df_te is None:
        print(f"[INFO] 无 {tag} 测试集，跳过。"); continue
    p_base = resolve_test_pred(tag)
    if p_base is None:
        print(f"[WARN] 未能解析 {tag} 的基础预测，跳过。"); continue
    meta_te = ensure_meta(df_te, tag)
    # A) seg isotonic
    p_iso = apply_segment_isotonic(p_base, meta_te, iso_models, iso_global, seg_cols_used)
    save_sub(df_te, p_iso, os.path.join(OUT_DIR, f"test_{tag}_pred_segment_isotonic.csv"))
    # A-rank) rank 融合
    p_rank = best_w*to_rank01(p_base) + (1-best_w)*to_rank01(p_iso)
    save_sub(df_te, p_rank, os.path.join(OUT_DIR, f"test_{tag}_pred_rankblend_raw_iso.csv"))
    # B) stack LR
    p_stack = apply_stack_lr(p_base, meta_te, stack_model, stack_cols)
    save_sub(df_te, p_stack, os.path.join(OUT_DIR, f"test_{tag}_pred_stack_lr.csv"))


/opt/anaconda3/envs/LoanCom/lib/python3.7/site-packages/ipykernel_launcher.py:132: FutureWarning: casting datetime64[ns, UTC] values to int64 with .astype(...) is deprecated and will raise in a future version. Use .view(...) instead.


[INFO] 使用分段列： ['grade', 'has_stm']
[INFO] OOF base AUC = 0.6640302545603807
[A] Segment Isotonic OOF AUC = 0.668020938010953
[A-rank] best_w=0.00 | OOF AUC=0.668021
[B] Stack LR OOF AUC = 0.6646174978626463
[WARN] 未能解析 aa 的基础预测，跳过。


/opt/anaconda3/envs/LoanCom/lib/python3.7/site-packages/ipykernel_launcher.py:132: FutureWarning: casting datetime64[ns, UTC] values to int64 with .astype(...) is deprecated and will raise in a future version. Use .view(...) instead.


[SAVE] outputs_postprocess_v1/test_ab_pred_segment_isotonic.csv
[SAVE] outputs_postprocess_v1/test_ab_pred_rankblend_raw_iso.csv
[SAVE] outputs_postprocess_v1/test_ab_pred_stack_lr.csv
